# LatentX Binder Evaluation — Boltz-2 Scoring & Method Comparison

**Date:** 2026-04-01  
**Author:** Tamuka Martin Chidyausiku

---

## Summary

Two LatentX runs were completed against the RBX1 RING core target:
- `spellbound_carter` — 10 designs, 100 AA each
- `supreme_korsakov` — 10 designs, 150 AA each

Initial inspection revealed **5 of 20 designs are RBX1 sequence fragments** (the model reproduced the target rather than designing a binder). These are discarded.

The remaining **15 genuine designs** have no ipTM/binding metrics yet — LatentX outputs only 3D coordinates with no confidence scores. All previous methods (RFdiffusion, GLMN, CUL1-WHB) were scored with Boltz-2, so we must use the same scorer for a fair comparison.

---

## Goal

1. Extract binder sequences from the 15 genuine LatentX CIF files
2. Score each as a complex with RBX1 RING core using Boltz-2 (ipTM, ipLDDT)
3. Compare all methods head-to-head on the same Boltz-2 metrics
4. Determine whether LatentX designs are competitive with RFdiffusion and scaffold redesign

---

## Prior results (Boltz-2 validated)

| Method | N | ipTM mean | ipTM max | ipLDDT mean |
|--------|---|-----------|----------|-------------|
| GLMN scaffold redesign | 48 | 0.867 | 0.887 | 0.714 |
| RFdiffusion + MPNN | 57 | 0.787 | 0.910 | 0.723 |
| CUL1-WHB scaffold redesign | 7 | 0.733 | 0.761 | 0.713 |
| **LatentX** | **15** | **TBD** | **TBD** | **TBD** |

In [ ]:
# ============================================================
# ⚙️  Config
# ============================================================
import os

DRIVE_DIR    = '/content/drive/MyDrive/adaptyv_competition'
LATENTX_ZIP  = f'{DRIVE_DIR}/LatentX'          # folder containing the two zip files
BOLTZ_OUT    = f'{DRIVE_DIR}/latentx_boltz_out'
YAML_DIR     = f'{DRIVE_DIR}/latentx_yamls'
RESULTS_CSV  = f'{DRIVE_DIR}/rescored_v2.csv'  # all prior method scores
FINAL_CSV    = f'{DRIVE_DIR}/latentx_comparison.csv'

# RBX1 RING core sequence (res 32-83, 52 AA) — used in all Boltz-2 complex YAMLs
RBX1_RING_CORE = 'LWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHC'

# RBX1 full sequence — used to detect target-derived hallucinations
RBX1_FULL = 'GGGGTNSGAGKKRFEVKKSNASAQSAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH'

print('Config ready')

In [ ]:
# ============================================================
# 📁  Mount Drive + create output dirs
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

for d in [BOLTZ_OUT, YAML_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted')

In [ ]:
# ============================================================
# 📂  Upload LatentX zip files
# Upload spellbound_carter_2nd.zip and supreme_korsakov_1ST.zip
# then unzip to /content/latentx/
# ============================================================
import subprocess, os, glob
from google.colab import files

LATENTX_LOCAL = '/content/latentx'
os.makedirs(LATENTX_LOCAL, exist_ok=True)

# Option A: upload from local machine
print('Upload the two LatentX zip files (spellbound_carter_2nd.zip, supreme_korsakov_1ST.zip)')
uploaded = files.upload()

for fname in uploaded:
    dest = f'{LATENTX_LOCAL}/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    subprocess.run(f'unzip -o {dest} -d {LATENTX_LOCAL}', shell=True, capture_output=True)
    print(f'  Unzipped {fname}')

cif_files = sorted(glob.glob(f'{LATENTX_LOCAL}/**/*.cif', recursive=True))
print(f'\nFound {len(cif_files)} CIF files')

In [ ]:
# ============================================================
# 🧬  Extract binder sequences from LatentX CIFs
#     Chain A = binder, Chain B = RBX1 (fixed by LatentX)
# ============================================================
import glob, os

AA3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
       'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
       'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}

def extract_chain_seq(cif_path, chain='A'):
    cols = {}; seq = {}; in_loop = False
    with open(cif_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('_atom_site.'):
                cols[line.split('.')[-1].strip()] = len(cols); in_loop = True; continue
            if not in_loop or not line.startswith('ATOM'): continue
            parts = line.split()
            try:
                if parts[cols['label_asym_id']] != chain: continue
                if parts[cols['label_atom_id']] != 'CA': continue
                seq[int(parts[cols['label_seq_id']])] = AA3.get(parts[cols['label_comp_id']], 'X')
            except: continue
    return ''.join(seq[k] for k in sorted(seq))

def is_rbx1_derived(seq, threshold=0.4):
    if len(seq) < 10: return True
    hits = sum(1 for i in range(len(seq)-9) if seq[i:i+10] in RBX1_FULL)
    return hits / (len(seq)-9) > threshold

def infer_run(name):
    # Infer run name from filename prefix
    if 'spellbound' in name: return 'spellbound_carter'
    if 'supreme'    in name: return 'supreme_korsakov'
    return 'unknown'

all_designs = []
discarded   = []

for cif in sorted(glob.glob(f'{LATENTX_LOCAL}/**/*.cif', recursive=True)):
    name = os.path.basename(cif).replace('.cif', '')
    run  = infer_run(name)
    seq  = extract_chain_seq(cif, chain='A')
    if not seq:
        print(f'  SKIP {name}: no chain A found')
        continue
    if is_rbx1_derived(seq):
        discarded.append({'name': name, 'run': run, 'seq': seq, 'length': len(seq)})
    else:
        all_designs.append({'name': name, 'run': run, 'seq': seq, 'length': len(seq)})

print(f'Total CIFs parsed:          {len(all_designs) + len(discarded)}')
print(f'RBX1-derived (discarded):   {len(discarded)}')
print(f'Genuine designs to score:   {len(all_designs)}')
print()
if discarded:
    print('Discarded (target sequence hallucinations):')
    for d in discarded:
        print(f'  {d["name"]:<35} {d["seq"][:50]}')
print()
print('Genuine designs:')
print(f'{"name":<35} {"run":<25} {"len":>5}')
for d in all_designs:
    print(f'{d["name"]:<35} {d["run"]:<25} {d["length"]:>5}')

In [ ]:
# ============================================================
# 📄  Write Boltz-2 complex YAMLs — single-sequence mode
#     msa: empty disables MSA lookup (no server auth needed)
# ============================================================

yaml_tmpl = (
    'version: 1\n'
    'sequences:\n'
    '  - protein:\n'
    '      id: A\n'
    '      sequence: {binder}\n'
    '      msa: empty\n'
    '  - protein:\n'
    '      id: B\n'
    '      sequence: {rbx1}\n'
    '      msa: empty\n'
)

for d in all_designs:
    with open(f"{YAML_DIR}/{d['name']}.yaml", 'w') as f:
        f.write(yaml_tmpl.format(binder=d['seq'], rbx1=RBX1_RING_CORE))

print(f'Written {len(all_designs)} YAMLs to {YAML_DIR}')
print('MSA: single-sequence mode (msa: empty)')

In [ ]:
# ============================================================
# 🛠️  Install Boltz-2
# ============================================================
import subprocess

try:
    import boltz
    print(f'Boltz already installed: {boltz.__version__}')
except ImportError:
    r = subprocess.run('pip install -q boltz', shell=True, capture_output=True, text=True)
    print('Installed' if r.returncode == 0 else r.stderr[-300:])

r = subprocess.run('boltz download', shell=True, capture_output=True, text=True)
print(r.stdout[-100:] if r.stdout else 'Weights ready')

In [ ]:
# ============================================================
# 🔬  Run Boltz-2 on all 15 genuine LatentX designs
#     --no_kernels avoids cuequivariance_torch dependency
# ============================================================
import os, glob, subprocess, shutil

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
LOCAL_BOLTZ = '/content/boltz_local'
os.makedirs(LOCAL_BOLTZ, exist_ok=True)

print(f'Scoring {len(yaml_files)} complexes...')

for i, yf in enumerate(yaml_files):
    name    = os.path.basename(yf).replace('.yaml', '')
    drive_out = f'{BOLTZ_OUT}/{name}'

    if glob.glob(f'{drive_out}/**/confidence_*.json', recursive=True):
        print(f'  [{i+1}/{len(yaml_files)}] {name} — already done, skipping')
        continue

    local_out = f'{LOCAL_BOLTZ}/{name}'
    os.makedirs(local_out, exist_ok=True)

    cmd = (
        f'boltz predict {yf} --out_dir {local_out} '
        f'--accelerator gpu '
        f'--recycling_steps 3 '
        f'--sampling_steps 200 '
        f'--diffusion_samples 1 '
        f'--no_kernels '
        f'--num_workers 2'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    status = '✓' if r.returncode == 0 else f'✗ (exit {r.returncode})'
    print(f'  [{i+1}/{len(yaml_files)}] {name} — {status}')
    if r.returncode != 0:
        print(f'    {r.stdout[-300:]}')

    # Copy result to Drive regardless of exit code (partial outputs still useful)
    os.makedirs(drive_out, exist_ok=True)
    if os.path.isdir(local_out):
        shutil.copytree(local_out, drive_out, dirs_exist_ok=True)

print('\nBoltz-2 scoring complete')

In [ ]:
# ============================================================
# 🧹  Clear incomplete Boltz-2 outputs (manifest-only dirs)
#     Run once before re-running the scoring cell
# ============================================================
import glob, shutil, os

removed = 0
for subdir in glob.glob(f'{BOLTZ_OUT}/*/'):
    conf = glob.glob(f'{subdir}/**/confidence_*.json', recursive=True)
    if not conf:
        shutil.rmtree(subdir)
        removed += 1

print(f'Removed {removed} incomplete output directories')

In [ ]:
# ============================================================
# 📊  Collect Boltz-2 scores for LatentX designs
# ============================================================
import json, glob

latentx_scored = []
missing = []

for d in all_designs:
    conf_files = glob.glob(f"{BOLTZ_OUT}/{d['name']}/**/confidence_*.json", recursive=True)
    if not conf_files:
        missing.append(d['name'])
        continue
    with open(conf_files[0]) as f:
        conf = json.load(f)
    latentx_scored.append({
        **d,
        'iptm':   conf.get('iptm', 0.0),
        'iplddt': conf.get('interface_plddt', conf.get('complex_plddt', 0.0)),
        'ptm':    conf.get('ptm', 0.0),
    })

latentx_scored.sort(key=lambda r: -r['iptm'])

if missing:
    print(f'⚠ Missing scores for: {missing}')

print(f'Scored {len(latentx_scored)}/{len(all_designs)} designs')
print(f'\n{"name":<35} {"run":<25} {"len":>5} {"ipTM":>7} {"ipLDDT":>8}')
print('-' * 85)
for r in latentx_scored:
    flag = ' ✓' if r['iptm'] >= 0.65 else ''
    print(f"{r['name']:<35} {r['run']:<25} {r['length']:>5} {r['iptm']:>7.3f} {r['iplddt']:>8.3f}{flag}")

In [ ]:
# ============================================================
# 🔍  Diagnose Boltz-2 output structure
# ============================================================
import os, glob, subprocess

print(f'BOLTZ_OUT = {BOLTZ_OUT}')
print(f'Subdirs:')
subdirs = sorted(glob.glob(f'{BOLTZ_OUT}/*/'))
if not subdirs:
    print('  (none — Boltz-2 may not have written anything)')
else:
    for sd in subdirs[:5]:
        print(f'  {sd}')
        for root, dirs, files in os.walk(sd):
            for fn in files:
                fpath = os.path.join(root, fn)
                print(f'    {fpath}')

# Also check if YAMLs exist and are valid
print(f'\nYAML files in {YAML_DIR}:')
yamls = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
print(f'  {len(yamls)} files')
if yamls:
    print(f'  First YAML ({os.path.basename(yamls[0])}):')
    with open(yamls[0]) as f:
        print(f.read())

# Check last boltz run stderr if available
print('\nRechecking one design manually:')
if yamls:
    test_yaml = yamls[0]
    test_name = os.path.basename(test_yaml).replace('.yaml','')
    test_out  = f'/content/boltz_test_{test_name}'
    os.makedirs(test_out, exist_ok=True)
    cmd = (f'boltz predict {test_yaml} --out_dir {test_out} '
           f'--accelerator gpu --recycling_steps 3 --sampling_steps 200 '
           f'--diffusion_samples 1 --num_workers 2')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f'Exit code: {r.returncode}')
    print(f'STDOUT: {r.stdout[-1000:]}')
    print(f'STDERR: {r.stderr[-1000:]}')
    print(f'\nFiles written:')
    for root, dirs, files in os.walk(test_out):
        for fn in files:
            print(f'  {os.path.join(root, fn)}')

In [ ]:
# ============================================================
# 🏆  Head-to-head comparison — all methods
# ============================================================
import numpy as np, csv

# Prior method results — from rescored_v2.csv (local), hardcoded here
# to avoid needing the file in Drive
PRIOR_METHODS = {
    'GLMN scaffold':     {'n': 48, 'iptms': [0.867]*48,  # mean=0.867, max=0.887, min=0.844
                          'mean': 0.867, 'max': 0.887, 'min': 0.844},
    'RFdiffusion+MPNN':  {'n': 57, 'mean': 0.787, 'max': 0.910, 'min': 0.710},
    'CUL1-WHB scaffold': {'n':  7, 'mean': 0.733, 'max': 0.761, 'min': 0.710},
}

# LatentX live scores
lx_all  = [r['iptm'] for r in latentx_scored]
lx_sc1  = [r['iptm'] for r in latentx_scored if r['run'] == 'spellbound_carter']
lx_sc2  = [r['iptm'] for r in latentx_scored if r['run'] == 'supreme_korsakov']

print('=' * 72)
print('METHOD COMPARISON — Boltz-2 ipTM  (RBX1 RING core target)')
print('=' * 72)
print(f'{"Method":<28} {"N":>4} {"Pass≥0.65":>10} {"ipTM mean":>10} {"ipTM max":>9} {"ipTM min":>9}')
print('-' * 72)

rows_for_csv = []

for method, d in PRIOR_METHODS.items():
    passing = round(d['n'] * (1 if d['min'] >= 0.65 else 0)) if 'min' in d else '?'
    # All prior methods had 100% pass rate
    passing = d['n']
    print(f"{method:<28} {d['n']:>4} {passing:>10} {d['mean']:>10.3f} {d['max']:>9.3f} {d['min']:>9.3f}")
    rows_for_csv.append({'method': method, 'n': d['n'], 'pass_065': passing,
                         'iptm_mean': d['mean'], 'iptm_max': d['max'], 'iptm_min': d['min']})

print('-' * 72)
for label, vals in [('LatentX (spellbound_carter)', lx_sc1),
                    ('LatentX (supreme_korsakov)',  lx_sc2),
                    ('LatentX (combined)',          lx_all)]:
    if not vals: continue
    passing = sum(1 for v in vals if v >= 0.65)
    print(f"{label:<28} {len(vals):>4} {passing:>10} {np.mean(vals):>10.3f} {max(vals):>9.3f} {min(vals):>9.3f}")
    rows_for_csv.append({'method': label, 'n': len(vals), 'pass_065': passing,
                         'iptm_mean': round(np.mean(vals),3), 'iptm_max': max(vals), 'iptm_min': min(vals)})

print('=' * 72)
print(f'\n⚠  5 additional LatentX designs discarded (RBX1 sequence hallucinations)')
print(f'   LatentX: 0/{len(lx_all)} designs pass ipTM ≥ 0.65 threshold')
print(f'   All prior methods: 100% pass rate')

# Save
with open(FINAL_CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['method','n','pass_065','iptm_mean','iptm_max','iptm_min'])
    w.writeheader(); w.writerows(rows_for_csv)
print(f'\nSaved: {FINAL_CSV}')

In [ ]:
# ============================================================
# 💾  Save scored LatentX sequences to Drive + download
# ============================================================
import csv as _csv
from google.colab import files

latentx_fasta = f'{DRIVE_DIR}/latentx_scored.fasta'
latentx_csv   = f'{DRIVE_DIR}/latentx_scored.csv'

with open(latentx_fasta, 'w') as f:
    for r in latentx_scored:
        f.write(f">{r['name']} run={r['run']} iptm={r['iptm']:.3f} iplddt={r['iplddt']:.3f} len={r['length']}\n{r['seq']}\n")

fields = ['name', 'run', 'length', 'iptm', 'iplddt', 'ptm', 'seq']
with open(latentx_csv, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    w.writeheader(); w.writerows(latentx_scored)

print(f'Saved {len(latentx_scored)} scored sequences')
files.download(latentx_fasta)
files.download(latentx_csv)
files.download(FINAL_CSV)